# Complementarity and MPEC Models

Many equilibrium, bilevel, and contact problems include **complementarity**
conditions

$$ 0 \le f(x) \;\perp\; g(x) \ge 0 \qquad\Longleftrightarrow\qquad f \ge 0,\; g \ge 0,\; f\cdot g = 0. $$

A program with such constraints is a *Mathematical Program with Equilibrium
Constraints* (MPEC). The product condition $f\cdot g = 0$ is nonconvex and
violates the standard constraint qualifications, so a naive NLP solve stalls
{cite:p}`LuoPangRalph1996`.

`discopt.mpec` provides the two standard reformulations:

1. **Scholtes regularization** {cite:p}`Scholtes2001` — relax $f\cdot g = 0$ to
   $f\cdot g \le t$ and drive $t \to 0$ through a homotopy of *local* NLP solves.
2. **SOS1 / disjunctive** — encode "at most one of $f$, $g$ is nonzero" exactly
   with a Special Ordered Set of type 1, solved by the global MINLP
   branch-and-bound. This is the discrete analogue of solving the MPEC as an NLP
   {cite:p}`FletcherLeyffer2004`.

Both build an ordinary discopt model, so the global-optimality machinery is
reused unchanged.


In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_ENABLE_X64"] = "1"

import numpy as np
import discopt.modeling.core as dm
from discopt.mpec import complementarity, solve_mpec, tighten_complementarity_bounds


## A reference MPEC

We minimize the distance from $(x, y)$ to $(1, 1)$ subject to
$0 \le x \perp y \ge 0$. The complementarity forces $x\cdot y = 0$, so the
global optima sit at $(1, 0)$ and $(0, 1)$, both with objective $1$.


In [ ]:
def distance_model():
    m = dm.Model("mpec_distance")
    x = m.continuous("x", lb=0, ub=10)
    y = m.continuous("y", lb=0, ub=10)
    m.minimize((x - 1) ** 2 + (y - 1) ** 2)
    return m, x, y


## Scholtes regularization

The homotopy solves a sequence of smooth NLPs with $t = 1, 10^{-1}, \dots$,
warm-starting each from the previous solution. As $t \to 0$ the iterate
approaches the complementary set.


In [ ]:
m, x, y = distance_model()
res = solve_mpec(m, [complementarity(x, y)], method="scholtes")
print("x, y =", np.round(res.x, 6))
print("objective =", round(float(res.objective), 6))
print("x*y =", float(res.x[0] * res.x[1]))


The iterate converges to $(0, 1)$ (up to the regularization tolerance) with
objective $1$ and $x\cdot y \approx 0$ — a global optimum of the MPEC.


## Exact SOS1 reformulation

For a certifiable answer we encode the complementarity exactly. The SOS1 set
$\{x, y\}$ allows at most one member to be nonzero; the global MINLP solver
branches on the two disjuncts $x = 0$ and $y = 0$.


In [ ]:
m2, x2, y2 = distance_model()
res2 = solve_mpec(m2, [complementarity(x2, y2)], method="sos1")
print("status   =", res2.status)
print("objective =", round(float(res2.objective), 6))
# Model.solve returns a name -> value mapping.
print("x, y =", float(np.asarray(res2.x["x"])), float(np.asarray(res2.x["y"])))


Both reformulations reach objective $1$, confirming the same global optimum
by two independent routes.


## Complementarity bound propagation

When one side of $0 \le x \perp y \ge 0$ is provably strictly positive
($\text{lb} > 0$), the other side must be zero. This sound implication is
exact for single-variable sides and tightens the model before search.


In [ ]:
m3 = dm.Model("bp")
a = m3.continuous("a", lb=0.5, ub=3.0)   # strictly positive lower bound
b = m3.continuous("b", lb=0.0, ub=3.0)
n_fixed = tighten_complementarity_bounds(m3, [complementarity(a, b)])
print("variables fixed to zero:", n_fixed)
print("b bounds:", float(b.lb), float(b.ub))


## When to use which

| Method | Path | Best for |
|--------|------|----------|
| `scholtes` | homotopy of local NLPs | smooth continuous MPECs; fast |
| `sos1` | global MINLP branch-and-bound | exact certificates; small/medium discrete blocks |

Scholtes is the workhorse for large continuous equilibrium models; the SOS1
route provides a global certificate and pairs naturally with discopt's
branch-and-bound when the complementary blocks are modest in number.
